# 02 - Sentiment validation and session alignment

The PRD asks for hand-labelled headlines from *our own* corpus. Financial PhraseBank is excluded because FinBERT was trained on it. Labels come from the text only, never from subsequent price moves. Fill the `human_label` column of `reports/sentiment_labelling.csv` with `positive`, `negative` or `neutral`; this notebook then reports accuracy, macro-F1 and confusion matrices for FinBERT and VADER.

In [1]:
import sys
from pathlib import Path

root = Path.cwd()
while not (root / 'src').exists() and root != root.parent:
    root = root.parent
sys.path.insert(0, str(root))

import pandas as pd
from src.utils import load_config, path_from_config
from src.sentiment import evaluate_labels
from src.clean import nyse_calendar
from src.sentiment import session_dates

cfg = load_config()
sheet = pd.read_csv(path_from_config(cfg, 'reports') / 'sentiment_labelling.csv')
models = pd.read_csv(path_from_config(cfg, 'reports') / 'sentiment_labelling_models.csv')
sheet = sheet.merge(models, on='id', how='left')
labelled = sheet['human_label'].astype(str).str.lower().isin(['positive', 'negative', 'neutral']).sum()
print(f'{len(sheet)} headlines, {labelled} labelled')
sheet.head(8)

200 headlines, 200 labelled


,id,ticker,timestamp,title,human_label,notes,finbert_label,finbert_score,vader
0,1,HD,2020-02-07 00:00:00+00:00,"Salomon & Ludwin, LLC Buys SPDR S&P 5…… ETF Tr...",neutral,unsure,neutral,0.871119,0.5106
1,2,PG,2015-12-29 00:00:00+00:00,A Look At Some Of 2015's Exciting S&P 500 ETFs,neutral,NaN,neutral,0.932489,0.4939
2,3,NVDA,2022-11-07 00:00:00+00:00,3 Stocks Down 50% or More That Are Screaming B...,negative,NaN,negative,0.964543,-0.4336
3,4,AAPL,2023-03-09 00:00:00+00:00,Should BNY Mellon US Large Cap Core Equity ETF...,neutral,NaN,neutral,0.875932,0.0000
4,5,KO,2019-08-20 00:00:00+00:00,8 Stocks With Unbelievable Dividend Growth,positive,NaN,neutral,0.919950,0.5267
5,6,KO,2015-01-22 00:00:00+00:00,Can You Name Warren Buffett's Top 5 Stock Hold...,neutral,NaN,neutral,0.945147,0.2023
6,7,XOM,2022-01-25 00:00:00+00:00,Top 5 S&P 500 Stocks Poised to Beat on Q4 Earn...,positive,NaN,positive,0.844641,0.4215
7,8,JNJ,2020-02-14 00:00:00+00:00,The Daily Biotech Pulse: Epizyme's Tazemetosta...,neutral,NaN,neutral,0.625583,-0.0516


## Protocol

- 200 headlines stratified by ticker and year.
- Labelled from the text only; never from the next-day return (that would leak the target).
- One person labelled every headline. The optional 50-headline re-label (Cohen's kappa) was not done.
- VADER is a lexicon baseline; FinBERT is finance-tuned. Neither is assumed correct.

In [2]:
if labelled:
    result = evaluate_labels(cfg)
    display(result)
else:
    print('Labels are pending. Fill human_label in reports/sentiment_labelling.csv, then re-run.')
    print('Until then the report states the validation status as pending, never as a result.')

{'status': 'ok',
 'rows': 200,
 'flagged_unsure': 1,
 'finbert': {'accuracy': 0.67,
  'accuracy_ci': [0.605, 0.735],
  'macro_f1': 0.6466314200304348,
  'confusion': {'negative': {'negative': 27, 'neutral': 3, 'positive': 10},
   'neutral': {'negative': 12, 'neutral': 74, 'positive': 28},
   'positive': {'negative': 8, 'neutral': 5, 'positive': 33}}},
 'vader': {'accuracy': 0.445,
  'accuracy_ci': [0.375, 0.515],
  'macro_f1': 0.43472052223582214,
  'confusion': {'negative': {'negative': 15, 'neutral': 6, 'positive': 8},
   'neutral': {'negative': 17, 'neutral': 42, 'positive': 31},
   'positive': {'negative': 15, 'neutral': 34, 'positive': 32}}},
 'finbert_minus_vader_ci': [0.139875, 0.31]}

In [3]:
examples = sheet.sample(10, random_state=cfg['project']['seed'])[['ticker', 'timestamp', 'title', 'finbert_label', 'finbert_score', 'vader']]
examples

,ticker,timestamp,title,finbert_label,finbert_score,vader
95,JPM,2019-01-16 00:00:00+00:00,"Dimon doesn't see a hard Brexit, but is still ...",neutral,0.840703,0.2846
15,MSFT,2023-07-26 00:00:00+00:00,"1 Green Flag for Microsoft in 2023, and 1 Red ...",neutral,0.803662,0.0000
30,MSFT,2022-06-02 00:00:00+00:00,"US STOCKS-Wall Street rallies, lifted by Tesla...",positive,0.865649,0.0000
158,AAPL,2020-04-30 00:00:00+00:00,12 Stocks Moving In Thursday's After-Hours Ses...,neutral,0.725561,0.0000
128,CAT,2018-08-16 00:00:00+00:00,How Will Caterpillar Perform In Second Half Of...,neutral,0.757881,0.7096
115,JPM,2018-10-23 00:00:00+00:00,"Zelle chalks up $32B, 116M in payments in Q3",neutral,0.747639,0.0000
69,CAT,2019-10-08 00:00:00+00:00,Caterpillar or Deere: Which Heavy-Equipment Ma...,neutral,0.893984,0.4404
170,KO,2015-03-18 00:00:00+00:00,"Dow Movers: KO, CAT",neutral,0.876355,0.0000
174,XOM,2021-06-14 00:00:00+00:00,AMC Entertainment's Moves Make Sense,neutral,0.874310,0.4215
45,XOM,2022-03-21 00:00:00+00:00,Should WisdomTree U.S. Total Dividend ETF (DTD...,neutral,0.927434,0.0000


## Session-alignment checks

A headline at 16:00 New York time or later, a date-only stamp and a weekend/holiday headline must all inform the next session, not the session that has already closed.

In [4]:
cal = nyse_calendar('2020-01-01', '2020-12-31')
ts = pd.Series(pd.to_datetime([
    '2020-01-10 20:59:00',  # Friday 15:59 ET
    '2020-01-10 21:00:00',  # Friday 16:00 ET
    '2020-01-11 15:00:00',  # Saturday
    '2020-07-03 15:00:00',  # Independence Day observed
    '2020-03-09 20:00:00',  # DST: Monday 16:00 EDT
], utc=True))
mapped = session_dates(ts, cal)
pd.DataFrame({'utc': ts, 'session': mapped})

,utc,session
0,2020-01-10 20:59:00+00:00,2020-01-10
1,2020-01-10 21:00:00+00:00,2020-01-13
2,2020-01-11 15:00:00+00:00,2020-01-13
3,2020-07-03 15:00:00+00:00,2020-07-06
4,2020-03-09 20:00:00+00:00,2020-03-10


The expected rows are Friday 10th, Monday 13th, Monday 13th, Monday 6th July and Tuesday 10th March. DST is handled by converting to America/New_York before applying the 16:00 rule, not by shifting UTC by a fixed offset.